## Feature Engineering for Fraud Detection: Location Anomalies

Location anomalies are a common indicator of fraud. This example demonstrates how to create features that can help detect such anomalies by analyzing transaction locations and times.

### 1. Generate Sample Transaction Data

Let's create some synthetic transaction data including `user_id`, `timestamp`, `latitude`, and `longitude`.

In [10]:
import pandas as pd
import numpy as np

# 1. Define the Haversine formula to calculate distance in kilometers
def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Differences in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Core Haversine calculation
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))

    # Earth's radius in km is approximately 6371
    return 6371 * c

# 2. Simplified Mock transaction data with 6 complete rows
data = {
    "tx_id": [1, 2, 3, 4, 5, 6],
    "timestamp": pd.to_datetime([
        "2026-05-18 10:00:00",
        "2026-05-18 10:05:00",
        "2026-05-18 12:00:00",
        "2026-05-18 12:30:00",  # 4. Local London commute
        "2026-05-18 14:00:00",  # 5. Flight to Paris (1.5 hours later)
        "2026-05-18 14:05:00"   # 6. Impossible travel to Tokyo (5 mins later)
    ]),
    "lat": [40.7128, 51.5074, 51.5090, 51.5048, 48.8566, 35.6762],  # NYC -> London -> London -> London -> Paris -> Tokyo
    "lon": [-74.0060, -0.1278, -0.1260, -0.1522, 2.3522, 139.6503],
}
df = pd.DataFrame(data)
df

,tx_id,timestamp,lat,lon
0,1,2026-05-18 10:00:00,40.7128,-74.0060
1,2,2026-05-18 10:05:00,51.5074,-0.1278
2,3,2026-05-18 12:00:00,51.5090,-0.1260
3,4,2026-05-18 12:30:00,51.5048,-0.1522
4,5,2026-05-18 14:00:00,48.8566,2.3522
5,6,2026-05-18 14:05:00,35.6762,139.6503


In [11]:

# 3. Fetch coordinates and timestamps from the preceding transaction directly
df["prev_lat"] = df["lat"].shift(1)
df["prev_lon"] = df["lon"].shift(1)
df["prev_time"] = df["timestamp"].shift(1)

# 4. Engineer the distance, time difference, and required speed metrics
df["distance_km"] = haversine_distance(df["lat"], df["lon"], df["prev_lat"], df["prev_lon"])
df["hours_diff"] = (df["timestamp"] - df["prev_time"]).dt.total_seconds() / 3600.0
df["required_speed_kph"] = df["distance_km"] / df["hours_diff"]

# 5. Apply thresholds (Flag if distance > 1000km AND required speed > 900km/h)
df["impossible_travel"] = (df["distance_km"] > 1000) & (df["required_speed_kph"] > 900)

print(df[["tx_id", "distance_km", "required_speed_kph", "impossible_travel"]])


   tx_id  distance_km  required_speed_kph  impossible_travel
0      1          NaN                 NaN              False
1      2  5570.222180        66842.666157               True
2      3     0.217190            0.113316              False
3      4     1.872477            3.744954              False
4      5   344.206504          229.471002              False
5      6  9711.724819       116540.697823               True
